# Tópicos Avançados em IA — Aula 01
## Séries temporais: componentes e decomposição

Prof. James Moraes de Almeida

Roteiro:
1. O que é uma série temporal, e por que ela não é uma tabela comum.
2. Os componentes: tendência, sazonalidade, ciclo e resíduo.
3. Modelos aditivo e multiplicativo — e o logaritmo que passa de um para o outro.
4. Três formas de estimar a tendência: regressão, filtro de Hodrick-Prescott e médias móveis.
5. A decomposição clássica, passo a passo, conferida contra o `statsmodels`.
6. Uma série real: a rupia indiana. Ela tem sazonalidade?
7. Para ir além, e exercícios.

Ambiente: `conda activate topicos-ia` (arquivo `environment.yml` na pasta da disciplina).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import spsolve
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.seasonal import seasonal_decompose

plt.rcParams["figure.figsize"] = (11, 3.2)

## 1. O que é uma série temporal

Uma série temporal é uma sequência de observações $y_1, y_2, \dots, y_T$ de uma mesma
grandeza, feitas em instantes **igualmente espaçados** — a cada hora, dia, mês. Formalmente,
é **uma realização** de um processo estocástico $\{Y_t\}$: a série que temos é um dos
caminhos possíveis, como uma única rodada de um experimento que não pode ser repetido.

O que a separa de uma tabela comum de aprendizado de máquina é a **dependência no tempo**.
Numa tabela, supomos as linhas independentes e trocáveis de ordem. Numa série, $y_t$ se
parece com $y_{t-1}$: embaralhar as linhas destrói exatamente a informação que queremos
usar. Tudo nesta disciplina — da divisão treino/teste (Aula 02) aos modelos — decorre disso.

Dois números descrevem o calendário da série:
* a **frequência** $f$: observações por ano (12 para mensal, 4 para trimestral);
* o **período sazonal** $m$: depois de quantas observações o padrão se repete
  (12 para um padrão anual em dados mensais; 24 para um padrão diário em dados horários).

## 2. Os componentes

A decomposição supõe que a série é a combinação de partes com significados diferentes.
Usamos a mesma notação dos slides:

| símbolo | componente | o que é |
|---|---|---|
| $\tau_t$ | tendência | o movimento de longo prazo: crescimento, queda, mudança de patamar |
| $c_t$ | sazonalidade | um padrão que se repete com **período fixo e conhecido** $m$: $c_{t+m} = c_t$ |
| $\varepsilon_t$ | resíduo | o que sobra; idealmente sem estrutura |

Um quarto componente aparece na literatura: o **ciclo** — oscilações **sem** período fixo,
como os ciclos econômicos, que duram de 2 a 10 anos sem regularidade. A diferença importa:
sazonalidade tem calendário (todo dezembro), ciclo não. Na decomposição clássica o ciclo fica
junto com a tendência; no filtro de Hodrick-Prescott, o "ciclo" é tudo o que não é tendência.

**Uma convenção necessária:** a sazonalidade soma zero ao longo de um período,
$\sum_{j=1}^{m} c_j = 0$. Sem isso, somar uma constante a $c_t$ e tirá-la de $\tau_t$ daria a
mesma série — os dois componentes não seriam identificáveis.

### Construindo uma série com componentes conhecidos

Numa série real não sabemos os componentes verdadeiros. Numa sintética sabemos — e é assim que
podemos **conferir** se um método de decomposição funciona. Doze anos de dados mensais:

$$\tau_t = 100 + 2{,}65\,t, \qquad c_t = 20\,\operatorname{sen}\!\left(\frac{2\pi t}{12}\right), \qquad
\varepsilon_t \sim \mathcal{N}(0,\ 3^2).$$

Repare no $2\pi t/12$: é ele que faz o período ser **exatamente** 12 meses. Com
$\operatorname{sen}(0{,}5\,t)$, por exemplo, o período seria $4\pi \approx 12{,}6$ meses — e decompor com
período 12 procuraria um padrão que a série não tem.

In [ ]:
rng = np.random.default_rng(0)
t = np.arange(144)                              # meses
m = 12                                          # período sazonal
tendencia = 100 + 2.65 * t
sazonal = 20 * np.sin(2 * np.pi * t / m)
residuo = rng.normal(0, 3, len(t))

print(f"soma da sazonalidade num período: {sazonal[:m].sum():.1e}")

fig, eixos = plt.subplots(3, 1, figsize=(11, 6), sharex=True)
for ax, serie, nome, cor in zip(eixos, [tendencia, sazonal, residuo],
                                ["tendência τ", "sazonalidade c", "resíduo ε"],
                                ["tab:red", "tab:orange", "tab:green"]):
    ax.plot(t, serie, color=cor); ax.set_ylabel(nome); ax.grid(alpha=0.3)
eixos[-1].set_xlabel("tempo (meses)")
plt.tight_layout()

## 3. Modelos aditivo e multiplicativo

**Aditivo:** $y_t = \tau_t + c_t + \varepsilon_t$. A sazonalidade tem amplitude **fixa**, na
unidade da série: "em dezembro, 20 unidades a mais".

**Multiplicativo:** $y_t = \tau_t \cdot s_t \cdot e_t$. Agora $s_t$ e $e_t$ são **fatores
relativos, em torno de 1**: "em dezembro, 20% a mais". Como o efeito é proporcional ao nível,
a oscilação cresce junto com a tendência. É o caso típico de vendas, passageiros e consumo.

Um erro comum é multiplicar a série por um resíduo de **média zero**: metade dos fatores fica
negativa e a série vira ruído — os resíduos "dominam". No modelo multiplicativo o erro precisa
oscilar em torno de 1, não de 0: $e_t = 1 + \eta_t$, com $\eta_t$ pequeno.

In [ ]:
fator_sazonal = 1 + 0.2 * np.sin(2 * np.pi * t / m)     # ±20%
fator_erro = 1 + rng.normal(0, 0.02, len(t))              # ±2%
aditiva = tendencia + sazonal + residuo
multiplicativa = tendencia * fator_sazonal * fator_erro

fig, eixos = plt.subplots(1, 2, figsize=(13, 3.5))
eixos[0].plot(t, aditiva); eixos[0].set_title("aditiva: amplitude fixa")
eixos[1].plot(t, multiplicativa, color="tab:purple"); eixos[1].set_title("multiplicativa: amplitude cresce")
for ax in eixos:
    ax.set_xlabel("tempo (meses)"); ax.set_ylabel("valor (u.a.)"); ax.grid(alpha=0.3)
plt.tight_layout()

def amplitude(z, ano):
    fatia = z[12 * ano:12 * (ano + 1)]
    return fatia.max() - fatia.min()

# amplitude da oscilação em torno da tendência verdadeira, no 1º e no 12º ano
for nome, desvio in [("aditiva: y − τ", aditiva - tendencia),
                     ("multiplicativa: y − τ", multiplicativa - tendencia),
                     ("multiplicativa: log y − log τ", np.log(multiplicativa) - np.log(tendencia))]:
    print(f"{nome:32s} 1º ano {amplitude(desvio, 0):7.2f}   12º ano {amplitude(desvio, 11):7.2f}")

### O logaritmo transforma multiplicativo em aditivo

$$\log y_t = \log \tau_t + \log s_t + \log e_t .$$

Na última linha da tabela acima, a oscilação de $\log y$ em torno de $\log \tau$ tem
praticamente a mesma amplitude no primeiro e no último ano — a pequena diferença é o ruído de 2% —,
enquanto na escala original (linha do meio) ela cresce mais de quatro vezes. Por isso, diante de uma série multiplicativa, o caminho
mais comum é **tirar o logaritmo, modelar como aditiva e voltar com a exponencial** — é o que
a Atividade ST da Aula 02 pede com a série de passageiros aéreos.

Para $\eta$ pequeno, $\log(1 + \eta) \approx \eta$: o erro relativo de 2% vira um erro aditivo
de ~0,02 na escala do log.

## 4. Estimando a tendência

### 4.1 Regressão linear

Se a tendência é uma reta, $\tau_t = a + b\,t$, os mínimos quadrados dão
$\hat b = \dfrac{\sum_t (t - \bar t)(y_t - \bar y)}{\sum_t (t - \bar t)^2}$ e
$\hat a = \bar y - \hat b\,\bar t$. O que sobra, $y_t - \hat a - \hat b t$, é a série
**sem tendência**: sazonalidade mais resíduo.

In [ ]:
b, a = np.polyfit(t, aditiva, 1)
reta = a + b * t
print(f"inclinação estimada {b:.3f} por mês (verdadeira 2,65) · intercepto {a:.2f} (verdadeiro 100)")

plt.plot(t, aditiva, label="série")
plt.plot(t, reta, label="tendência linear")
plt.plot(t, aditiva - reta, label="sem tendência = c + ε")
plt.xlabel("tempo (meses)"); plt.ylabel("valor (u.a.)"); plt.legend(); plt.grid(alpha=0.3);

Funciona **porque construímos a tendência linear**. Em série real a tendência raramente é
uma reta — e uma reta ajustada a uma curva deixa parte da tendência no que deveria ser
sazonalidade e resíduo.

### 4.2 O filtro de Hodrick-Prescott

O filtro HP (Hodrick e Prescott, 1997) escolhe a tendência $\tau$ que resolve

$$\min_{\tau}\ \underbrace{\sum_{t=1}^{T} (y_t - \tau_t)^2}_{\text{ficar perto dos dados}}
\;+\; \lambda \underbrace{\sum_{t=2}^{T-1} \big[(\tau_{t+1} - \tau_t) - (\tau_t - \tau_{t-1})\big]^2}_{\text{não mudar de inclinação}} .$$

O segundo termo é a soma das **segundas diferenças** — a "curvatura" da tendência. O
parâmetro $\lambda$ (o `lamb` do `statsmodels`) decide o peso de cada termo:

* $\lambda \to 0$: a tendência copia a série, $\tau = y$;
* $\lambda \to \infty$: a curvatura tem de ser zero, e a tendência vira **uma reta** — a de
  mínimos quadrados da seção 4.1.

**A solução tem forma fechada.** Com $D$ a matriz $(T-2)\times T$ das segundas diferenças
(cada linha tem $1, -2, 1$), o problema é $\min_\tau \|y - \tau\|^2 + \lambda\|D\tau\|^2$.
Derivando e igualando a zero: $(I + \lambda D^\top D)\,\tau = y$. É um sistema linear —
podemos resolvê-lo à mão e conferir contra o `statsmodels`.

Atenção à ordem: `hpfilter` devolve **(ciclo, tendência)**, nessa ordem, e o ciclo é
$y - \tau$ — **tudo** o que não é tendência: sazonalidade e resíduo juntos. O HP não separa a
sazonalidade.

In [ ]:
def hp_a_mao(y, lam):
    T = len(y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, 1, 2], shape=(T - 2, T), format="csc")
    return spsolve((sparse.eye(T, format="csc") + lam * (D.T @ D)).tocsc(), np.asarray(y, float))

ciclo, tend_hp = hpfilter(aditiva, lamb=1600)
print(f"HP à mão vs statsmodels:              maior diferença {np.abs(hp_a_mao(aditiva, 1600) - tend_hp).max():.1e}")
print(f"HP com λ = 10¹⁰ vs reta de mínimos q.: maior diferença {np.abs(hp_a_mao(aditiva, 1e10) - reta).max():.1e}")

plt.plot(t, aditiva, label="série", alpha=0.6)
plt.plot(t, tend_hp, label="tendência HP (λ = 1600)")
plt.plot(t, ciclo, label="ciclo = série − tendência (inclui a sazonalidade!)")
plt.xlabel("tempo (meses)"); plt.ylabel("valor (u.a.)"); plt.legend(); plt.grid(alpha=0.3);

### Como escolher λ?

$\lambda$ **não é estimado**: é uma escolha de suavidade, e a convenção vem da economia.

* Hodrick e Prescott propuseram $\lambda = 1600$ para dados **trimestrais**.
* Ravn e Uhlig (2002) mostraram como ajustar para outra frequência $f$ (observações por ano):
  $\lambda = 1600\,(f/4)^4$. Para dados **mensais**, $f = 12$: $\lambda = 1600 \cdot 3^4 = 129\,600$.
  Para anuais, $f = 1$: $\lambda = 6{,}25$.

Uma forma de ver a potência 4: o termo de curvatura usa segundas diferenças **ao quadrado**.
Com $k$ vezes mais observações por ano, cada segunda diferença de uma mesma curva fica $k^2$
vezes menor, e o seu quadrado $k^4$ vezes menor — λ precisa crescer $k^4$ para pesar o mesmo.

Nossa série é mensal. Como a tendência verdadeira é conhecida, podemos medir o erro de cada λ:

In [ ]:
# as tendências ficam quase sobrepostas no gráfico da série; o erro contra a verdadeira mostra a diferença
fig, ax = plt.subplots()
for lam in [10, 1600, 129_600, 1e7]:
    tr = hp_a_mao(aditiva, lam)
    ax.plot(t, tr - tendencia, label=f"λ = {lam:g}")
    print(f"λ = {lam:>10g}: maior erro da tendência HP contra a verdadeira = {np.abs(tr - tendencia).max():5.2f} u.a.")
ax.axhline(0, color="k", lw=0.8)
ax.set_title("erro da tendência HP: estimada − verdadeira")
ax.set_xlabel("tempo (meses)"); ax.set_ylabel("erro (u.a.)"); ax.legend(ncol=4, fontsize=9); ax.grid(alpha=0.3);

Com λ pequeno a "tendência" persegue a sazonalidade. O valor trimestral (1600) aplicado a dados
mensais ainda deixa a tendência ondular com a sazonalidade; o valor mensal de Ravn-Uhlig chega
perto da verdadeira. Numa série real não há "verdadeira" para comparar: a escolha segue a
convenção da frequência, e se declara no relatório.

Um alerta da literatura: Hamilton (2018) mostrou que o HP cria dinâmicas espúrias e é pouco
confiável **nas pontas** da série — justamente onde se quer prever. Ele é bom para descrever o
passado, não para alimentar previsões.

### 4.3 Médias móveis

A média móvel de ordem $k$ troca cada ponto pela média dos vizinhos. Duas versões:

* **Atrasada** (o que o `np.convolve(..., "valid")` faz): $\dfrac{1}{k}\sum_{j=0}^{k-1} y_{t-j}$.
  Usa só o passado — mas o valor calculado com os meses 0 a 11 "pertence" ao mês 5,5, e não ao
  mês 11. Desenhada contra o índice errado, ela aparece **atrasada** em relação à série.
* **Centrada:** $\dfrac{1}{k}\sum_{j=-h}^{h} y_{t+j}$, com $k = 2h + 1$ ímpar.

**Por que a média de 12 meses remove a sazonalidade:** ela soma exatamente um período
completo de $c_t$, e a sazonalidade soma zero num período. Sobra só a tendência (mais o resíduo
suavizado).

**E se o período é par?** Doze meses não têm um centro. A solução é a média **2×12**: a média
de duas médias de 12 deslocadas de um mês, o que dá 13 termos com pesos
$\tfrac{1}{24}, \tfrac{1}{12}, \dots, \tfrac{1}{12}, \tfrac{1}{24}$. Ainda soma um período completo
e agora é centrada. É a que o `seasonal_decompose` usa.

In [ ]:
atrasada = np.convolve(aditiva, np.ones(12) / 12, mode="valid")   # 133 valores
pesos_2x12 = np.r_[0.5, np.ones(11), 0.5] / 12
centrada = np.convolve(aditiva, pesos_2x12, mode="same")
centrada[:6] = np.nan; centrada[-6:] = np.nan                        # sem vizinhos suficientes

plt.plot(t, aditiva, color="lightgray", label="série")
plt.plot(t[11:], atrasada, label="atrasada, desenhada no último mês da janela")
plt.plot(t, centrada, label="centrada 2×12")
plt.plot(t, tendencia, "k--", lw=1, label="tendência verdadeira")
plt.xlabel("tempo (meses)"); plt.ylabel("valor (u.a.)"); plt.legend(fontsize=9); plt.grid(alpha=0.3)

d_sm = seasonal_decompose(aditiva, model="additive", period=12)
print(f"2×12 à mão vs tendência do seasonal_decompose: maior diferença {np.nanmax(np.abs(centrada - d_sm.trend)):.1e}")
print(f"a atrasada fica, em média, {np.mean(tendencia[11:] - atrasada):.2f} u.a. abaixo da tendência "
      f"(5,5 meses × 2,65 por mês = {5.5 * 2.65:.2f})")

## 5. A decomposição clássica

Com a média móvel centrada, a decomposição aditiva clássica tem quatro passos:

1. **Tendência:** $\hat\tau_t$ = média móvel 2×$m$ (ou $m$, se ímpar).
2. **Tirar a tendência:** $d_t = y_t - \hat\tau_t$ — sobra sazonalidade mais resíduo.
3. **Sazonalidade:** para cada posição $j$ do período (cada mês do ano), a média de $d_t$ nos
   anos disponíveis, $\tilde c_j$. Depois **centrar**: $\hat c_j = \tilde c_j - \frac{1}{m}\sum_i \tilde c_i$,
   para somar zero. No modelo **multiplicativo**, o passo 2 é uma divisão ($y_t/\hat\tau_t$) e
   aqui os índices são **divididos** pela sua média, para terem média 1.
4. **Resíduo:** $\hat\varepsilon_t = y_t - \hat\tau_t - \hat c_t$.

Escrevemos os quatro passos à mão e conferimos contra o `seasonal_decompose`:

In [ ]:
d = aditiva - centrada                                                   # passo 2
c_til = np.array([np.nanmean(d[j::m]) for j in range(m)])               # passo 3
c_hat = np.tile(c_til - c_til.mean(), len(t) // m)
eps_hat = aditiva - centrada - c_hat                                     # passo 4

print(f"sazonalidade à mão vs statsmodels: maior diferença {np.abs(c_hat - d_sm.seasonal).max():.1e}")
print(f"resíduo à mão vs statsmodels:      maior diferença {np.nanmax(np.abs(eps_hat - d_sm.resid)):.1e}")
print(f"sazonalidade recuperada vs verdadeira: maior erro {np.abs(c_hat - sazonal).max():.2f} u.a. (amplitude 20)")
print(f"desvio padrão do resíduo: {np.nanstd(eps_hat):.2f} u.a. (o ruído verdadeiro tem σ = 3)")

fig, eixos = plt.subplots(4, 1, sharex=True, figsize=(11, 7))
for ax, (nome, est, verd) in zip(eixos, [("série", aditiva, None), ("tendência", centrada, tendencia),
                                         ("sazonalidade", c_hat, sazonal), ("resíduo", eps_hat, residuo)]):
    ax.plot(t, est, label="estimado")
    if verd is not None:
        ax.plot(t, verd, "k--", lw=0.8, label="verdadeiro")
    ax.set_ylabel(nome); ax.grid(alpha=0.3)
eixos[1].legend(loc="upper left", fontsize=9); eixos[-1].set_xlabel("tempo (meses)")
plt.tight_layout()

**Por que o resíduo sai menor que o ruído verdadeiro?** Porque a média móvel e as médias
por mês foram calculadas com os próprios dados e "absorvem" uma parte do ruído: o método
ajusta mais do que deveria. É um primeiro contato com o sobreajuste, que volta na Aula 02.

### E se errarmos o período?

In [ ]:
for p in [10, 11, 12, 13, 24]:
    r = seasonal_decompose(aditiva, model="additive", period=p).resid
    print(f"period = {p:2d}: desvio padrão do resíduo {np.nanstd(r):5.2f} u.a.")

Com o período errado (10, 11, 13) a sazonalidade não é removida e vai parar no resíduo.
Mas repare no **24**: o resíduo fica *menor* que com o período certo. Com 24 índices sazonais
em vez de 12, o método tem mais parâmetros livres e ajusta parte do ruído. **O resíduo não
serve para escolher o período** — o período vem do conhecimento do problema (12 meses, 7 dias,
24 horas), não de uma busca.

### Limitações da decomposição clássica

* Perde $m/2$ pontos em cada ponta (a média centrada não tem vizinhos lá) — exatamente onde se
  quer prever.
* Supõe a sazonalidade **igual todo ano**.
* É sensível a valores atípicos: um mês anômalo contamina a média daquela posição.

A alternativa moderna é o **STL** (Cleveland e colegas, 1990), que usa regressões locais,
deixa a sazonalidade mudar devagar e tem uma versão robusta a atípicos:
`statsmodels.tsa.seasonal.STL`.

### A série tem sazonalidade? A força da sazonalidade

O `seasonal_decompose` **sempre** devolve um componente sazonal: é uma suposição do método, não
uma descoberta. Para medir se ele significa alguma coisa, compara-se o tamanho do resíduo com o
tamanho da sazonalidade mais o resíduo (Wang, Smith e Hyndman, 2006):

$$F_S = \max\!\left(0,\ 1 - \frac{\operatorname{Var}(\hat\varepsilon)}{\operatorname{Var}(\hat c + \hat\varepsilon)}\right).$$

$F_S$ perto de 1: a sazonalidade explica quase toda a variação que sobra depois da tendência.
Perto de 0: o "componente sazonal" é do tamanho do ruído — não há sazonalidade de verdade.

In [ ]:
def forca_sazonal(decomp):
    r, s = np.asarray(decomp.resid), np.asarray(decomp.seasonal)
    return max(0.0, 1 - np.nanvar(r) / np.nanvar(s + r))

print(f"F_S da série sintética: {forca_sazonal(d_sm):.3f}")

## 6. Uma série real: a rupia indiana

Taxa de câmbio mensal, **rúpias por dólar**, de 1973 a 2019 (série EXINUS do Federal Reserve).
A tendência claramente não é linear.

A série é **mensal**. Comparamos o valor **trimestral**, $\lambda = 1600$, com o valor mensal de
Ravn-Uhlig, $\lambda = 129\,600$:

In [ ]:
rupia = pd.read_excel("data/India_Exchange_Rate_Dataset.xls", index_col=0, parse_dates=True)["EXINUS"]
print(f"{len(rupia)} meses, de {rupia.index.min():%m/%Y} a {rupia.index.max():%m/%Y}")

fig, eixos = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
eixos[0].plot(rupia, color="lightgray", label="rúpias por dólar")
for lam in [1600, 129_600]:
    ciclo_r, tend_r = hpfilter(rupia, lamb=lam)
    eixos[0].plot(tend_r, label=f"tendência HP, λ = {lam:g}")
    eixos[1].plot(ciclo_r, label=f"ciclo, λ = {lam:g}   (desvio padrão {ciclo_r.std():.2f} rúpia)")
eixos[0].set_ylabel("rúpias por dólar"); eixos[1].set_ylabel("ciclo (rúpias)")
for ax in eixos:
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()

Com λ maior a tendência fica mais rígida e o ciclo mais largo: mais variação é atribuída ao
"ciclo". Qual está certo? Nenhum dos dois é "o certo" — são escolhas de escala, e o valor mensal
é o que a convenção recomenda para dados mensais.

Agora a pergunta que importa: **a rupia tem sazonalidade?** Um período anual em dados mensais é
12; testamos também 24 (dois anos). Medimos os dois:

In [ ]:
for p in [12, 24]:
    dec = seasonal_decompose(rupia, model="additive", period=p)
    amp = dec.seasonal.max() - dec.seasonal.min()
    print(f"period = {p}: amplitude do componente sazonal {amp:.2f} rúpia · "
          f"desvio padrão da série {rupia.std():.2f} rúpias · F_S = {forca_sazonal(dec):.3f}")

dec = seasonal_decompose(rupia, model="additive", period=12)
fig, eixos = plt.subplots(4, 1, sharex=True, figsize=(11, 7))
for ax, (nome, s) in zip(eixos, [("série", rupia), ("tendência", dec.trend),
                                 ("sazonalidade", dec.seasonal), ("resíduo", dec.resid)]):
    ax.plot(s); ax.set_ylabel(nome); ax.grid(alpha=0.3)
eixos[0].set_title("Decomposição com período 12 — repare na escala do painel da sazonalidade")
plt.tight_layout()

**A decomposição devolveu uma "sazonalidade" — e ela não existe.** O componente sazonal tem
amplitude de fração de rúpia, contra um desvio padrão de 20 rúpias da série, e $F_S$ perto de
zero: é do tamanho do resíduo. O câmbio não tem um mês do ano sistematicamente mais caro.

A lição da aula: **todo método de decomposição devolve os componentes que supõe**. Decidir se
eles existem é trabalho de quem analisa — com um número, não com o olho.

---
## 7. Para ir além

**Por que a média de $m$ termos elimina uma sazonalidade de período $m$ — em frequência.** Uma
média móvel é um filtro linear. Aplicada a $\cos(\omega t)$, a média de $m$ termos multiplica a
amplitude por $\left|\frac{\operatorname{sen}(m\omega/2)}{m\,\operatorname{sen}(\omega/2)}\right|$, que é zero
quando $\omega = 2\pi k/m$ — a frequência fundamental da sazonalidade e todos os seus harmônicos.
Qualquer padrão de período $m$ é uma soma desses harmônicos, e some inteiro.

**A solução do HP como suavizador.** $\tau = (I + \lambda D^\top D)^{-1} y$ é uma média
ponderada de toda a série, com pesos que decaem para longe de cada ponto. Nas pontas os pesos
ficam assimétricos — é de onde vem o problema das pontas.

**Outros métodos que vale conhecer:** STL (`statsmodels.tsa.seasonal.STL`), X-13ARIMA-SEATS
(o método oficial de ajuste sazonal de vários institutos de estatística) e MSTL, para séries
com mais de uma sazonalidade — dados horários têm ciclo diário **e** semanal (Aula 03).

**Documentação:** [filtros de séries temporais do statsmodels](https://www.statsmodels.org/stable/tsa.html#time-series-filters).

### Exercícios (não valem conceito)
1. Decomponha `data/AirPassengers.csv` como aditiva e como multiplicativa. Calcule $F_S$ nas
   duas (na multiplicativa, no logaritmo dos componentes). Qual modelo descreve melhor?
2. Aplique o `STL` à série sintética com um valor atípico (some 100 u.a. a um mês). Compare com o
   `seasonal_decompose`, com e sem `robust=True`.
3. Agregue a série sintética em trimestres (média de 3 meses) e aplique o HP com $\lambda = 1600$.
   Compare com o HP mensal de $\lambda = 129\,600$. A regra de Ravn-Uhlig se confirma?

### Referências
* Hyndman, R. J.; Athanasopoulos, G. *Forecasting: principles and practice*, 3ª ed. OTexts, 2021.
  Gratuito em https://otexts.com/fpp3/ — os capítulos de decomposição cobrem esta aula inteira.
* Ferreira, P. C. *Análise de Séries Temporais*, 1ª ed. Elsevier, 2017 — em português, na pasta da
  disciplina.
* Hodrick, R. J.; Prescott, E. C. Postwar U.S. business cycles: an empirical investigation.
  *Journal of Money, Credit and Banking*, 29(1), 1997.
* Ravn, M. O.; Uhlig, H. On adjusting the Hodrick-Prescott filter for the frequency of observations.
  *Review of Economics and Statistics*, 84(2), 2002.
* Hamilton, J. D. Why you should never use the Hodrick-Prescott filter. *Review of Economics and
  Statistics*, 100(5), 2018.
* Cleveland, R. B. e colegas. STL: a seasonal-trend decomposition procedure based on loess.
  *Journal of Official Statistics*, 6(1), 1990.
* Wang, X.; Smith, K.; Hyndman, R. J. Characteristic-based clustering for time series data.
  *Data Mining and Knowledge Discovery*, 13, 2006.
* Leitura de apoio, na pasta da disciplina: Vishwas e Patel, *Hands-on Time Series Analysis with
  Python* (2020); Atwan, *Time Series Analysis with Python Cookbook* (2024); Aloorravi, *Mastering
  Time Series Analysis and Forecasting with Python* (2024).